# k-NN y Perceptrón: memorizar a los vecinos o resumir el mundo en un vector de pesos

**Ciencia de Datos, Sección A** · Sesión 11 · 27 de agosto de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

Dependencias: `pip install numpy matplotlib scikit-learn`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

## 1. k-NN desde cero

El algoritmo completo: calcular la distancia a todos los puntos conocidos, quedarse con los `k` más cercanos, y votar.

In [ ]:
def distancias(X, punto):
    """X: (n, d), punto: (d,) -> (n,) distancias euclidianas."""
    dif = X - punto          # broadcasting (n,d) - (d,)
    return np.sqrt((dif ** 2).sum(axis=1))


X = rng.normal(size=(500, 2))
d = distancias(X, np.array([1.0, 1.0]))
print(d.shape, d.min().round(3), d.max().round(3))

In [ ]:
def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for punto in X_test:
        d = distancias(X_train, punto)
        idx = np.argsort(d)[:k]        # los k más cercanos
        vecinos = y_train[idx]
        conteo = np.bincount(vecinos)
        preds.append(conteo.argmax())  # voto mayoritario
    return np.array(preds)

In [ ]:
AZUL, ROJO, GRIS, LINEA = "#3A6EA5", "#B04A2E", "#75808E", "#E0E4EA"

def eje_limpio(ax):
    ax.grid(color=LINEA, lw=0.6, alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

# Dos clases que se traslapan un poco, para que k importe
rng_v = np.random.default_rng(11)
Xv = np.vstack([rng_v.normal([-1.2, 0.0], 1.0, (60, 2)),
                rng_v.normal([1.2, 0.0], 1.0, (60, 2))])
yv = np.array([0] * 60 + [1] * 60)
colores_v = np.where(yv == 1, ROJO, AZUL)

consulta = np.array([0.2, 0.4])   # el punto nuevo
k = 7
dv = distancias(Xv, consulta)
idx = np.argsort(dv)[:k]
voto = np.bincount(yv[idx], minlength=2)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(Xv[:, 0], Xv[:, 1], c=colores_v, s=22, alpha=0.45)
for i in idx:
    ax.plot([consulta[0], Xv[i, 0]], [consulta[1], Xv[i, 1]], color=GRIS, lw=0.8)
ax.scatter(Xv[idx, 0], Xv[idx, 1], facecolor="none", edgecolor="black", s=100, lw=1.2,
           label=f"los {k} vecinos")
ax.add_patch(plt.Circle(consulta, dv[idx[-1]], fill=False, ls="--", color=GRIS, lw=1))
ax.scatter(*consulta, marker="*", s=280, color="black", zorder=6, label="punto nuevo")
ax.set_aspect("equal")
ax.set_title(f"k = {k}: votan {voto[0]} azules y {voto[1]} rojos, gana la clase {voto.argmax()}")
ax.legend(frameon=False, fontsize=8, loc="upper left")
eje_limpio(ax)
plt.tight_layout()
plt.show()

# Eso es TODO el algoritmo: medir, quedarse con los k más cercanos, contar.
# No hay pesos, no hay entrenamiento: el "modelo" es el dataset entero.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-4.5, 4.5, 90), np.linspace(-3.5, 3.5, 90))
malla = np.column_stack([xx.ravel(), yy.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
for ax, kk in zip(axes, [1, 7, 41]):
    Zk = knn_predict(Xv, yv, malla, k=kk).reshape(xx.shape)
    ax.contourf(xx, yy, Zk, levels=[-0.5, 0.5, 1.5], colors=[AZUL, ROJO], alpha=0.18)
    ax.scatter(Xv[:, 0], Xv[:, 1], c=colores_v, s=16)
    ax.set_title(f"k = {kk}")
    ax.set_aspect("equal")
fig.suptitle("k chico memoriza cada punto (frontera dentada); k grande promedia (frontera suave)")
plt.show()

# k = 1 dibuja islas alrededor de cada punto raro: overfitting.
# k = 41 casi ignora la estructura local: underfitting. El ejercicio 2
# encuentra el punto medio con la curva de accuracy.

## 2. La perilla k y por qué escalar es obligatorio

La distancia hereda las unidades de las variables: quien mide en números grandes se queda con el voto.

In [ ]:
# (ingreso mensual en quetzales, años de antigüedad)
A = np.array([8000.0, 2.0])
B = np.array([8300.0, 12.0])

print("sin escalar:", np.sqrt(((A - B) ** 2).sum()).round(2))

# escalado a mano (dividir cada variable por su rango típico)
escala = np.array([1000.0, 5.0])
print("escalado:   ", np.sqrt((((A - B) / escala) ** 2).sum()).round(2))

In [ ]:
# Clientes: (ingreso mensual en Q, años de antigüedad). La clase depende de la antigüedad.
ing = rng_v.normal(8000, 1500, 80)
ant = np.concatenate([rng_v.normal(3, 1.5, 40), rng_v.normal(12, 1.5, 40)])
Xc = np.column_stack([ing, ant])
yc = np.array([0] * 40 + [1] * 40)
q = np.array([8100.0, 11.0])   # un cliente nuevo, claramente del grupo 1

Xc_esc = (Xc - Xc.mean(axis=0)) / Xc.std(axis=0)
q_esc = (q - Xc.mean(axis=0)) / Xc.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, datos, punto, nombre in [(axes[0], Xc, q, "sin escalar: los vecinos los elige el ingreso"),
                                  (axes[1], Xc_esc, q_esc, "escalado: ahora cuentan las dos variables")]:
    dd = distancias(datos, punto)
    vec = np.argsort(dd)[:5]
    votos = np.bincount(yc[vec], minlength=2)
    ax.scatter(datos[:, 0], datos[:, 1], c=np.where(yc == 1, ROJO, AZUL), s=22, alpha=0.5)
    ax.scatter(datos[vec, 0], datos[vec, 1], facecolor="none", edgecolor="black", s=110, lw=1.3)
    ax.scatter(*punto, marker="*", s=260, color="black", zorder=6)
    ax.set_title(f"{nombre}\nvoto: {votos[0]} vs {votos[1]}", fontsize=10)
    eje_limpio(ax)
axes[0].set_xlabel("ingreso (Q)")
axes[0].set_ylabel("antigüedad (años)")
axes[1].set_xlabel("ingreso (z)")
axes[1].set_ylabel("antigüedad (z)")
plt.show()

# Sin escalar, una diferencia de 300 quetzales pesa más que 8 años de
# antigüedad, y los vecinos salen de cualquier clase. La distancia
# hereda las unidades: escalar no es opcional.

## 3. La maldición de la dimensionalidad

En dimensión alta, la distancia mínima y la máxima entre pares de puntos se parecen cada vez más: el concepto de "vecino" se vacía de contenido.

In [ ]:
for d_dim in [2, 10, 100, 1000]:
    P = rng.normal(size=(1000, d_dim))
    dif = P[:, None, :] - P[None, :, :]
    D = np.sqrt((dif ** 2).sum(axis=2))
    D = D[np.triu_indices(1000, k=1)]
    print(f"d={d_dim:5d}  min/max = {D.min() / D.max():.4f}")

In [ ]:
# Distancias entre todos los pares vía la identidad |a-b|^2 = |a|^2 + |b|^2 - 2 a.b
def distancias_pares(P):
    sq = (P ** 2).sum(axis=1)
    D2 = sq[:, None] + sq[None, :] - 2 * P @ P.T
    D = np.sqrt(np.clip(D2, 0, None))
    return D[np.triu_indices(len(P), k=1)]

dims = [2, 10, 100, 1000]
ratios = []
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6), layout="constrained")
for d_dim, color in zip(dims, [GRIS, AZUL, ROJO, "black"]):
    D = distancias_pares(rng_v.normal(size=(400, d_dim)))
    axes[0].hist(D / D.mean(), bins=60, density=True, histtype="step", lw=1.8,
                 color=color, label=f"d = {d_dim}")
    ratios.append(D.min() / D.max())
axes[0].set_xlabel("distancia / distancia promedio")
axes[0].set_ylabel("densidad")
axes[0].set_title("En dimensión alta todas las distancias se parecen")
axes[0].legend(frameon=False, fontsize=8)

axes[1].plot(dims, ratios, marker="o", color=AZUL, lw=2)
axes[1].set_xscale("log")
axes[1].set_xlabel("dimensión (escala log)")
axes[1].set_ylabel("distancia mínima / máxima")
axes[1].set_title("El vecino más cercano deja de ser cercano")
for ax in axes:
    eje_limpio(ax)
plt.show()

# La campana se angosta hasta ser un pico: el vecino más cercano está
# casi tan lejos como el más lejano, y "vecino" pierde su significado.

## 4. Lazy learning: k-NN en scikit-learn

El `Pipeline` escala usando **solo** el train, que es la forma de evitar leakage.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_iris, y_iris = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42)

modelo = make_pipeline(StandardScaler(),
                       KNeighborsClassifier(n_neighbors=5))
modelo.fit(X_train, y_train)
print("accuracy:", round(modelo.score(X_test, y_test), 3))

## 5. El Perceptrón

Modelo lineal más umbral: $z = w \cdot x + b$, y $\hat{y} = 1$ si $z > 0$.

Regla de aprendizaje: predecir, medir el error $e = y - \hat{y}$, y corregir $w \leftarrow w + \eta \, e \, x$.

In [ ]:
def perceptron(X, y, eta=0.1, epocas=50):
    w = np.zeros(X.shape[1])
    b = 0.0
    for ep in range(epocas):
        errores = 0
        for xi, yi in zip(X, y):
            pred = 1 if (w @ xi + b) > 0 else 0
            e = yi - pred
            w += eta * e * xi
            b += eta * e
            errores += abs(e)
        if errores == 0:
            return w, b, ep + 1
    return w, b, epocas

In [ ]:
from sklearn.datasets import make_blobs

Xb, yb = make_blobs(n_samples=200, centers=2, cluster_std=1.0, random_state=3)
w, b, eps = perceptron(Xb, yb)
print("pesos:", w.round(3), " sesgo:", round(b, 3), " épocas:", eps)

In [ ]:
def perceptron_pasos(X, y, eta=0.1, epocas=50):
    """Igual que perceptron(), pero guarda (w, b) después de cada corrección."""
    w = np.zeros(X.shape[1])
    b = 0.0
    historia = []
    for ep in range(epocas):
        errores = 0
        for xi, yi in zip(X, y):
            pred = 1 if (w @ xi + b) > 0 else 0
            e = yi - pred
            if e != 0:
                w = w + eta * e * xi
                b = b + eta * e
                errores += 1
                historia.append((w.copy(), b, xi.copy()))
        if errores == 0:
            break
    return historia

hist = perceptron_pasos(Xb, yb)
cuales = [0, 1, 2, len(hist) - 1]
xs = np.linspace(Xb[:, 0].min() - 1, Xb[:, 0].max() + 1, 100)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), layout="constrained")
for ax, j in zip(axes, cuales):
    w_j, b_j, x_culpable = hist[j]
    ax.scatter(Xb[:, 0], Xb[:, 1], c=np.where(yb == 1, ROJO, AZUL), s=14, alpha=0.5)
    if abs(w_j[1]) > 1e-9:
        ax.plot(xs, -(w_j[0] * xs + b_j) / w_j[1], color="black", lw=2)
    ax.scatter(*x_culpable, marker="x", s=90, color="black", lw=2, zorder=6)
    ax.set_xlim(Xb[:, 0].min() - 1, Xb[:, 0].max() + 1)
    ax.set_ylim(Xb[:, 1].min() - 1, Xb[:, 1].max() + 1)
    ax.set_title(f"corrección #{j + 1} de {len(hist)}", fontsize=10)
    eje_limpio(ax)
fig.suptitle("Cada error mueve la recta: la x marca el punto mal clasificado que causó esa corrección")
plt.show()

# La regla w <- w + eta * e * x empuja a w hacia x (si x era un 1 que
# quedó afuera) o lejos de x (si era un 0 que quedó adentro). La recta
# gira un poco con cada error y para cuando ya no hay errores.

In [ ]:
# Frontera de decisión final: w[0]*x1 + w[1]*x2 + b = 0
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.scatter(Xb[:, 0], Xb[:, 1], c=np.where(yb == 1, ROJO, AZUL), s=20, alpha=0.6)
xs = np.linspace(Xb[:, 0].min(), Xb[:, 0].max(), 100)
ax.plot(xs, -(w[0] * xs + b) / w[1], color="black", lw=2, label="frontera: w·x + b = 0")
# w es perpendicular a la frontera y apunta hacia la clase 1
medio = Xb.mean(axis=0)
ax.annotate("", xy=medio + 1.2 * w / np.linalg.norm(w), xytext=medio,
            arrowprops=dict(arrowstyle="->", color="black", lw=1.8))
ax.text(*(medio + 1.35 * w / np.linalg.norm(w)), "w", fontsize=11)
ax.set_title(f"Perceptrón: frontera lineal, convergió en {eps} épocas")
ax.legend(frameon=False, fontsize=8)
eje_limpio(ax)
plt.tight_layout()
plt.show()

## 6. El muro del XOR

No existe una recta que separe los unos de los ceros. El perceptrón no converge, y no por falta de iteraciones sino por su forma.

In [ ]:
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])

w_x, b_x, eps_x = perceptron(X_xor, y_xor, epocas=200)
print("épocas usadas:", eps_x, "(200 significa que nunca convergió)")

preds = np.array([1 if (w_x @ xi + b_x) > 0 else 0 for xi in X_xor])
print("predicciones:", preds, " reales:", y_xor)

In [ ]:
fig = plt.figure(figsize=(11, 4.2))

ax = fig.add_subplot(1, 2, 1)
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=np.where(y_xor == 1, ROJO, AZUL), s=200, zorder=5)
for xi, yi in zip(X_xor, y_xor):
    ax.text(xi[0] + 0.05, xi[1] + 0.06, str(yi), fontsize=12)
ax.axvline(0.5, color=GRIS, ls="--", lw=1.2)
ax.axhline(0.5, color=GRIS, ls="--", lw=1.2)
ax.plot([-0.2, 1.2], [1.2, -0.2], color=GRIS, ls="--", lw=1.2)
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_aspect("equal")
ax.set_title("XOR: toda recta deja un punto del lado equivocado")
eje_limpio(ax)

ax = fig.add_subplot(1, 2, 2, projection="3d")
z_xor = X_xor[:, 0] * X_xor[:, 1]        # una feature nueva: x1 * x2
ax.scatter(X_xor[:, 0], X_xor[:, 1], z_xor, c=np.where(y_xor == 1, ROJO, AZUL), s=120)
px, py = np.meshgrid(np.linspace(-0.2, 1.2, 10), np.linspace(-0.2, 1.2, 10))
ax.plot_surface(px, py, (px + py - 0.5) / 2, alpha=0.25, color=GRIS)
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_zlabel("$x_1 x_2$")
ax.set_title("Con la feature $x_1 x_2$, un plano basta")
ax.view_init(elev=22, azim=-50)
plt.tight_layout()
plt.show()

# El muro no es de épocas, es de forma: el perceptrón solo dibuja rectas.
# Agregar una feature (o una capa oculta, semana 13) levanta los puntos
# a un espacio donde sí son separables. Ese es el nacimiento de las redes.

## 7. Ejercicios

### Ejercicio 1: su k-NN contra el de sklearn

In [ ]:
# a) predecir con knn_predict (k=5) sobre X_test y medir accuracy
mi_pred = ...
mi_accuracy = ...

# b) comparar contra KNeighborsClassifier(5) SIN escalar (para que sea justo)
sk_accuracy = ...

# Verificación: deben coincidir casi exactamente
# print(mi_accuracy, sk_accuracy)

### Ejercicio 2: la curva de accuracy vs k

¿Dónde está el overfitting? ¿Y el underfitting? ¿Qué `k` eligen?

In [ ]:
ks = range(1, 41, 2)
acc_train, acc_test = [], []

for k in ks:
    # ¿Qué va aquí?
    # entrenar KNeighborsClassifier(k), evaluar en train y en test, guardar
    pass

# Graficar ambas curvas contra k
# fig, ax = plt.subplots()
# ax.plot(list(ks), acc_train, label="train")
# ax.plot(list(ks), acc_test, label="test")
# ax.legend(); plt.show()

### Ejercicio 3: contar las épocas del perceptrón

In [ ]:
# a) entrenar perceptron() sobre make_blobs(cluster_std=1.0) y ver en cuántas épocas para
# b) subir cluster_std a 4.0 (clases traslapadas): ¿converge?
#    modificar perceptron() para devolver la lista de errores por época y graficarla
# c) confirmar con el XOR que el problema no es el número de épocas

# ¿Qué va aquí?
...

## Lo esencial de hoy

- **k-NN**: no entrena, vota entre los `k` vecinos más cercanos
- `k` chico memoriza, `k` grande promedia; se elige con validación
- **Escalar es obligatorio**: la distancia hereda las unidades
- En **dimensión alta** todo queda equidistante y k-NN se degrada
- **Perceptrón**: modelo lineal con umbral; el error mueve los pesos
- Converge si los datos son separables; el **XOR** lo derrota

**Próxima clase (martes 1 de septiembre): Lab Proyecto 1.** Traigan su dataset y sus avances. Parcial I: jueves 3 de septiembre (Módulos 1 a 3).